In [1]:
pip install pandas scikit-learn tensorflow keras matplotlib seaborn emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.5 MB/s eta 0:00:00


MAIN CODE

In [3]:
# -------------------------
# 0. Mount Google Drive and Set Paths
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies if not done:
# pip install pandas scikit-learn tensorflow keras matplotlib seaborn emoji wordcloud

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
from wordcloud import WordCloud
from tabulate import tabulate

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, SimpleRNN, Bidirectional,
    Dense, Dropout, SpatialDropout1D, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# -------------------------
# 1. Load and Prepare Data
# -------------------------
# Update these paths to point to your actual files in Google Drive
blinkit_df = pd.read_csv("/content/drive/MyDrive/Capstone/blinkit_reviews.csv")
zepto_df = pd.read_csv("/content/drive/MyDrive/Capstone/zepto_reviews.csv")

# Add platform identifier
blinkit_df['platform'] = 'Blinkit'
zepto_df['platform'] = 'Zepto'

# Combine datasets
df = pd.concat([blinkit_df, zepto_df], ignore_index=True)

# -------------------------
# 2. Exploratory Data Analysis
# -------------------------
plt.figure(figsize=(12, 6))

# Rating distribution
plt.subplot(1, 2, 1)
sns.countplot(x='score', hue='platform', data=df, palette=['#FF6B6B', '#4ECDC4'])
plt.title('Rating Distribution by Platform')
plt.xlabel('Rating Score')
plt.ylabel('Count')

# Platform comparison
plt.subplot(1, 2, 2)
platform_counts = df['platform'].value_counts()
plt.pie(platform_counts, labels=platform_counts.index, autopct='%1.1f%%',
        colors=['#FF6B6B', '#4ECDC4'], startangle=90)
plt.title('Platform Distribution')
plt.tight_layout()
plt.savefig('eda_ratings_platform.png', dpi=300)
plt.show()

# -------------------------
# 3. Create Sentiment Labels
# -------------------------
# 0 = Negative, 1 = Neutral, 2 = Positive
df['sentiment'] = df['score'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

# Sentiment distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='sentiment', hue='platform', data=df, palette=['#FF6B6B', '#4ECDC4'])
plt.title('Sentiment Distribution by Platform')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks([0, 1, 2], ['Negative', 'Neutral', 'Positive'])
plt.legend(title='Platform')
plt.savefig('sentiment_distribution.png', dpi=300)
plt.show()


# -------------------------
# 4. Additional EDA Visuals
# -------------------------

# --- Review Length Distribution ---
df['review_length'] = df['review'].astype(str).apply(len)
df['word_count'] = df['review'].astype(str).apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='word_count', hue='platform', bins=40, kde=True, element="step", palette=['#FF6B6B', '#4ECDC4'])
plt.title('Word Count Distribution by Platform')
plt.xlabel('Word Count')
plt.ylabel('Density')
plt.savefig('word_count_distribution.png', dpi=300)
plt.show()

# --- Average Rating per Platform ---
avg_ratings = df.groupby('platform')['score'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(x='platform', y='score', data=avg_ratings, palette=['#FF6B6B', '#4ECDC4'])
plt.title('Average Rating per Platform')
plt.ylabel('Average Score')
plt.savefig('avg_rating_platform.png', dpi=300)
plt.show()

# --- Sentiment vs Rating ---
plt.figure(figsize=(10, 6))
sns.boxplot(x='sentiment', y='score', hue='platform', data=df, palette=['#FF6B6B', '#4ECDC4'])
plt.title('Rating Distribution within Sentiment Groups')
plt.xlabel('Sentiment (0=Neg, 1=Neutral, 2=Pos)')
plt.ylabel('Rating Score')
plt.savefig('sentiment_vs_rating.png', dpi=300)
plt.show()

# --- Class Imbalance Check ---
sentiment_counts = df['sentiment'].value_counts().sort_index()
plt.figure(figsize=(6, 5))
sns.barplot(x=['Negative', 'Neutral', 'Positive'], y=sentiment_counts.values, palette='viridis')
plt.title('Class Distribution (All Platforms)')
plt.ylabel('Count')
plt.savefig('class_distribution.png', dpi=300)
plt.show()

# --- WordCloud per Sentiment ---
for sentiment, label in zip([0, 1, 2], ['Negative', 'Neutral', 'Positive']):
    text = " ".join(df[df['sentiment'] == sentiment]['review'].dropna().astype(str))
    wordcloud = WordCloud(width=800, height=400, background_color="white", colormap="coolwarm").generate(text)
    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud - {label} Reviews")
    plt.savefig(f'wordcloud_{label.lower()}.png', dpi=300)
    plt.show()

# --- Top Emoji Usage ---
def extract_emojis(text):
    return [c for c in text if c in emoji.EMOJI_DATA]

df['emojis'] = df['review'].astype(str).apply(extract_emojis)
emoji_list = sum(df['emojis'].tolist(), [])
emoji_freq = pd.Series(emoji_list).value_counts().head(15)

if not emoji_freq.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=emoji_freq.values, y=emoji_freq.index, palette="mako")
    plt.title("Top 15 Most Used Emojis in Reviews")
    plt.xlabel("Frequency")
    plt.ylabel("Emoji")
    plt.savefig('emoji_usage.png', dpi=300)
    plt.show()

MessageError: Error: credential propagation was unsuccessful

OLD

In [ ]:
# -------------------------
# 4. Text Preprocessing
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_content'] = df['content'].apply(clean_text)

# Generate word clouds
def generate_wordcloud(sentiment, platform):
    text = " ".join(df[(df['sentiment'] == sentiment) &
                      (df['platform'] == platform)]['clean_content'])
    wordcloud = WordCloud(width=800, height=400,
                          background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f"{platform} - {['Negative', 'Neutral', 'Positive'][sentiment]} Words")
    plt.axis('off')
    plt.savefig(f'wordcloud_{platform}_{sentiment}.png', dpi=300)
    plt.show()

for platform in ['Zomato', 'Swiggy']:
    for sentiment in [0, 1, 2]:
        generate_wordcloud(sentiment, platform)

# -------------------------
# 5. Tokenization & Padding
# -------------------------
MAX_WORDS = 20000  # Increased vocabulary size
MAX_LEN = 250  # Increased sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_content'])

X = tokenizer.texts_to_sequences(df['clean_content'])
X = pad_sequences(X, maxlen=MAX_LEN, padding='post', truncating='post')

y = np.array(df['sentiment'])
platforms = df['platform'].values

# -------------------------
# 6. Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test, platforms_train, platforms_test = train_test_split(
    X, y, platforms, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 7. Enhanced Model Architectures
# -------------------------
# def create_model(rnn_type, bidirectional=False):
#     model = Sequential()
#     model.add(Embedding(MAX_WORDS, 300, input_length=MAX_LEN))  # Increased embedding dimension
#     model.add(SpatialDropout1D(0.4))  # Increased dropout

#     # First RNN layer
#     if bidirectional:
#         if rnn_type == 'LSTM':
#             model.add(Bidirectional(LSTM(192, return_sequences=True,
#                                         kernel_regularizer=l2(0.02),
#                                         recurrent_dropout=0.3)))  # Added recurrent dropout
#         elif rnn_type == 'GRU':
#             model.add(Bidirectional(GRU(192, return_sequences=True,
#                                        kernel_regularizer=l2(0.02),
#                                        recurrent_dropout=0.3)))
#         else:  # RNN
#             model.add(Bidirectional(SimpleRNN(192, return_sequences=True,
#                                               kernel_regularizer=l2(0.02),
#                                               recurrent_dropout=0.3)))
#     else:
#         if rnn_type == 'LSTM':
#             model.add(LSTM(192, return_sequences=True,
#                            kernel_regularizer=l2(0.02),
#                            recurrent_dropout=0.3))
#         elif rnn_type == 'GRU':
#             model.add(GRU(192, return_sequences=True,
#                           kernel_regularizer=l2(0.02),
#                           recurrent_dropout=0.3))
#         else:  # RNN
#             model.add(SimpleRNN(192, return_sequences=True,
#                                 kernel_regularizer=l2(0.02),
#                                 recurrent_dropout=0.3))

#     model.add(Dropout(0.5))  # Increased dropout
#     model.add(BatchNormalization())

#     # Second RNN layer
#     if bidirectional:
#         if rnn_type == 'LSTM':
#             model.add(Bidirectional(LSTM(128, return_sequences=True,  # Added return_sequences
#                                         kernel_regularizer=l2(0.01),
#                                         recurrent_dropout=0.2)))
#         elif rnn_type == 'GRU':
#             model.add(Bidirectional(GRU(128, return_sequences=True,
#                                        kernel_regularizer=l2(0.01),
#                                        recurrent_dropout=0.2)))
#         else:  # RNN
#             model.add(Bidirectional(SimpleRNN(128, return_sequences=True,
#                                               kernel_regularizer=l2(0.01),
#                                               recurrent_dropout=0.2)))
#     else:
#         if rnn_type == 'LSTM':
#             model.add(LSTM(128, return_sequences=True,
#                            kernel_regularizer=l2(0.01),
#                            recurrent_dropout=0.2))
#         elif rnn_type == 'GRU':
#             model.add(GRU(128, return_sequences=True,
#                           kernel_regularizer=l2(0.01),
#                           recurrent_dropout=0.2))
#         else:  # RNN
#             model.add(SimpleRNN(128, return_sequences=True,
#                                 kernel_regularizer=l2(0.01),
#                                 recurrent_dropout=0.2))

#     model.add(Dropout(0.4))
#     model.add(BatchNormalization())

#     # Third RNN layer
#     if bidirectional:
#         if rnn_type == 'LSTM':
#             model.add(Bidirectional(LSTM(64)))
#         elif rnn_type == 'GRU':
#             model.add(Bidirectional(GRU(64)))
#         else:  # RNN
#             model.add(Bidirectional(SimpleRNN(64)))
#     else:
#         if rnn_type == 'LSTM':
#             model.add(LSTM(64))
#         elif rnn_type == 'GRU':
#             model.add(GRU(64))
#         else:  # RNN
#             model.add(SimpleRNN(64))

#     model.add(Dropout(0.3))
#     model.add(BatchNormalization())

#     # Dense layers
#     model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.01)))  # More units
#     model.add(Dropout(0.4))
#     model.add(BatchNormalization())
#     model.add(Dense(64, activation='relu'))
#     model.add(Dropout(0.3))
#     model.add(Dense(32, activation='relu'))
#     model.add(Dense(3, activation='softmax'))

#     optimizer = Adam(learning_rate=0.0005)  # Lower learning rate
#     model.compile(loss='sparse_categorical_crossentropy',
#                   optimizer=optimizer,
#                   metrics=['accuracy'])
#     return model
def create_model(rnn_type, bidirectional=False):
    model = Sequential()
    model.add(Embedding(MAX_WORDS, 128, input_length=MAX_LEN))  # Reduced from 300
    model.add(SpatialDropout1D(0.3))  # Reduced from 0.4

    # Single RNN layer instead of three
    if bidirectional:
        if rnn_type == 'LSTM':
            model.add(Bidirectional(LSTM(64, return_sequences=False,  # Reduced from 192
                                      kernel_regularizer=l2(0.01))))  # Reduced from 0.02
        elif rnn_type == 'GRU':
            model.add(Bidirectional(GRU(64, return_sequences=False,
                                      kernel_regularizer=l2(0.01))))
        else:  # RNN
            model.add(Bidirectional(SimpleRNN(64, return_sequences=False,
                                           kernel_regularizer=l2(0.01))))
    else:
        if rnn_type == 'LSTM':
            model.add(LSTM(64, return_sequences=False,
                         kernel_regularizer=l2(0.01)))
        elif rnn_type == 'GRU':
            model.add(GRU(64, return_sequences=False,
                        kernel_regularizer=l2(0.01)))
        else:  # RNN
            model.add(SimpleRNN(64, return_sequences=False,
                             kernel_regularizer=l2(0.01)))

    model.add(Dropout(0.3))  # Reduced from 0.5
    model.add(BatchNormalization())

    # Simplified dense layers
    model.add(Dense(64, activation='relu'))  # Reduced from 128
    model.add(Dropout(0.2))  # Reduced from 0.4
    model.add(Dense(3, activation='softmax'))  # Removed one dense layer

    optimizer = Adam(learning_rate=0.001)  # Increased from 0.0005
    model.compile(loss='sparse_categorical_crossentropy',
                 optimizer=optimizer,
                 metrics=['accuracy'])
    return model

# Handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, min_lr=1e-6)

# Models to train
models = {
    "RNN": create_model('RNN'),
    "BiRNN": create_model('RNN', bidirectional=True),
    "LSTM": create_model('LSTM'),
    "BiLSTM": create_model('LSTM', bidirectional=True),
    "GRU": create_model('GRU'),
    "BiGRU": create_model('GRU', bidirectional=True)
}

# -------------------------
# 8. Train & Evaluate Models
# -------------------------
results = {"Zomato": {}, "Swiggy": {}, "Overall": {}}
histories = {}
platform_predictions = {"Zomato": [], "Swiggy": []}
classification_reports = {}
final_metrics = []

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name} model...")
    print(f"{'='*50}")

    # Train the model
    # history = model.fit(
    #     X_train, y_train,
    #     epochs=30,  # Increased epochs
    #     batch_size=256,
    #     validation_split=0.1,
    #     class_weight=class_weights,
    #     callbacks=[early_stop, reduce_lr],
    #     verbose=1
    # )
    # Adjust these in your training code:
    history = model.fit(
    X_train, y_train,
    epochs=25,  # Reduced from 30
    batch_size=512,  # Increased from 256
    validation_split=0.1,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
    histories[name] = history

    # Evaluate on test set
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results["Overall"][name] = test_acc
    print(f"\n{name} Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

    # Predictions
    y_pred = np.argmax(model.predict(X_test), axis=1)

    # Platform-specific evaluation
    for platform in ["Zomato", "Swiggy"]:
        mask = (platforms_test == platform)
        platform_X = X_test[mask]
        platform_y = y_test[mask]

        if len(platform_X) > 0:
            _, acc = model.evaluate(platform_X, platform_y, verbose=0)
            results[platform][name] = acc
            platform_predictions[platform].extend(y_pred[mask])

            # Store classification reports
            report = classification_report(
                platform_y, y_pred[mask],
                target_names=['Negative', 'Neutral', 'Positive'],
                output_dict=True
            )
            classification_reports[f"{platform}_{name}"] = report

            print(f"{name} {platform} Accuracy: {acc:.4f}")

            # Store metrics for final summary
            final_metrics.append({
                "Model": name,
                "Platform": platform,
                "Accuracy": acc,
                "Precision": report['weighted avg']['precision'],
                "Recall": report['weighted avg']['recall'],
                "F1-Score": report['weighted avg']['f1-score']
            })

# -------------------------
# 9. Enhanced Visualization & Analysis
# -------------------------
# 1. Model accuracy comparison
plt.figure(figsize=(14, 8))
df_results = pd.DataFrame(results)
sns.barplot(data=df_results.reset_index().melt(id_vars='index'),
            x='index', y='value', hue='variable')
plt.title("Model Accuracy Comparison by Platform", fontsize=16)
plt.ylabel("Accuracy", fontsize=12)
plt.xlabel("Model", fontsize=12)
plt.ylim(0.7, 1.0)
plt.xticks(rotation=15)
plt.legend(title="Platform", title_fontsize=12)
plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=300)
plt.show()

# 2. Sentiment distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
platform_names = ["Zomato", "Swiggy"]
colors = ['#FF6B6B', '#4ECDC4', '#1A936F']  # Colors for sentiments

for i, platform in enumerate(platform_names):
    counts = [platform_predictions[platform].count(j) for j in range(3)]
    axes[i].bar(['Negative', 'Neutral', 'Positive'], counts, color=colors)
    axes[i].set_title(f'{platform} Sentiment Distribution', fontsize=14)
    axes[i].set_xlabel('Sentiment', fontsize=12)
    axes[i].set_ylabel('Count', fontsize=12)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

    # Add percentage labels
    total = sum(counts)
    for j, count in enumerate(counts):
        height = count
        axes[i].text(j, height + 0.01*total, f'{count/total:.1%}',
                    ha='center', va='bottom', fontsize=12)

plt.suptitle("Platform Sentiment Distributions", fontsize=16)
plt.tight_layout()
plt.savefig('platform_sentiment_distributions.png', dpi=300)
plt.show()

# 3. Training history comparison (Accuracy)
plt.figure(figsize=(14, 10))
plt.subplot(2, 1, 1)
for name, history in histories.items():
    plt.plot(history.history['accuracy'], '--', label=f'{name} Train')
    plt.plot(history.history['val_accuracy'], '-', label=f'{name} Val')
plt.title('Training vs Validation Accuracy', fontsize=16)
plt.ylabel('Accuracy', fontsize=12)
plt.xlabel('Epoch', fontsize=12)
plt.legend()
plt.grid(True)

# Training history comparison (Loss)
plt.subplot(2, 1, 2)
for name, history in histories.items():
    plt.plot(history.history['loss'], '--', label=f'{name} Train')
    plt.plot(history.history['val_loss'], '-', label=f'{name} Val')
plt.title('Training vs Validation Loss', fontsize=16)
plt.ylabel('Loss', fontsize=12)
plt.xlabel('Epoch', fontsize=12)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('model_training_history_comparison.png', dpi=300)
plt.show()

# 4. Individual model training history
for name, history in histories.items():
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{name} Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{name} Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f'training_history_{name}.png', dpi=300)
    plt.show()

# 5. Confusion matrices
best_models = {
    "Zomato": max(results["Zomato"], key=results["Zomato"].get),
    "Swiggy": max(results["Swiggy"], key=results["Swiggy"].get)
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for i, platform in enumerate(["Zomato", "Swiggy"]):
    model_name = best_models[platform]
    mask = (platforms_test == platform)
    y_true = y_test[mask]
    y_pred = np.argmax(models[model_name].predict(X_test[mask]), axis=1)

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'],
                ax=axes[i])
    axes[i].set_title(f'{platform} ({model_name})\nAccuracy: {results[platform][model_name]:.4f}', fontsize=14)
    axes[i].set_xlabel('Predicted', fontsize=12)
    axes[i].set_ylabel('Actual', fontsize=12)

plt.suptitle('Best Model Confusion Matrices by Platform', fontsize=16)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300)
plt.show()

# -------------------------
# 10. Enhanced Summary Report
# -------------------------
print("\n" + "="*50)
print("FINAL SUMMARY REPORT")
print("="*50)

# Best model for each platform
for platform in ["Zomato", "Swiggy"]:
    best_model = best_models[platform]
    accuracy = results[platform][best_model]
    print(f"\n{platform} Best Model: {best_model} (Accuracy: {accuracy:.4f})")

    # Sentiment percentages
    counts = [platform_predictions[platform].count(j) for j in range(3)]
    total = sum(counts)
    print(f"Sentiment Distribution: Negative: {counts[0]/total:.2%}, "
          f"Neutral: {counts[1]/total:.2%}, Positive: {counts[2]/total:.2%}")

# Model accuracy summary table
print("\n" + "="*50)
print("MODEL ACCURACY SUMMARY")
print("="*50)

summary_data = []
for model_name in models.keys():
    row = [model_name]
    row.append(f"{results['Overall'].get(model_name, 0):.4f}")
    row.append(f"{results['Zomato'].get(model_name, 0):.4f}")
    row.append(f"{results['Swiggy'].get(model_name, 0):.4f}")
    summary_data.append(row)

print(tabulate(summary_data,
               headers=["Model", "Overall Accuracy", "Zomato Accuracy", "Swiggy Accuracy"],
               tablefmt="grid"))

# Detailed metrics table
print("\n" + "="*50)
print("DETAILED MODEL PERFORMANCE METRICS")
print("="*50)

metrics_df = pd.DataFrame(final_metrics)
metrics_df = metrics_df.sort_values(by=['Platform', 'Accuracy'], ascending=[True, False])
print(tabulate(metrics_df, headers='keys', tablefmt='grid'))

# Save predictions for further analysis
output_df = df.loc[X_test.index].copy()
output_df['predicted_sentiment'] = np.argmax(models[best_models['Zomato']].predict(X_test), axis=1)
output_df.to_csv('sentiment_predictions.csv', index=False)

print("\nAnalysis completed. All visualizations saved.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ================================
# 1. Create DataFrame for metrics
# ================================
data = [
    # Swiggy
    ["RNN", "Swiggy", 0.778, 0.881, 0.778, 0.810],
    ["BiRNN", "Swiggy", 0.858, 0.892, 0.858, 0.873],
    ["LSTM", "Swiggy", 0.559, 0.312, 0.559, 0.400],
    ["BiLSTM", "Swiggy", 0.842, 0.897, 0.842, 0.865],
    ["GRU", "Swiggy", 0.559, 0.312, 0.559, 0.400],
    ["BiGRU", "Swiggy", 0.854, 0.896, 0.854, 0.872],

    # Zomato
    ["RNN", "Zomato", 0.872, 0.918, 0.872, 0.889],
    ["BiRNN", "Zomato", 0.900, 0.925, 0.900, 0.911],
    ["LSTM", "Zomato", 0.823, 0.678, 0.823, 0.744],
    ["BiLSTM", "Zomato", 0.888, 0.928, 0.888, 0.906],
    ["GRU", "Zomato", 0.823, 0.678, 0.823, 0.744],
    ["BiGRU", "Zomato", 0.897, 0.928, 0.897, 0.911],
]

df = pd.DataFrame(data, columns=["Model", "Platform", "Accuracy", "Precision", "Recall", "F1"])

# ================================
# 2. Show Table
# ================================
print("\n===== Detailed Metrics Table =====\n")
print(df.to_string(index=False))

# ================================
# 3. Plot Graphs
# ================================

# Separate graphs for Zomato & Swiggy
for platform in ["Zomato", "Swiggy"]:
    df_platform = df[df["Platform"] == platform]

    df_platform.plot(
        x="Model",
        y=["Accuracy", "Precision", "Recall", "F1"],
        kind="bar",
        figsize=(10, 6)
    )
    plt.title(f"{platform} - Model Performance Comparison")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=0)
    plt.legend(loc="lower right")
    plt.show()

# ================================
# 4. Combined Graph
# ================================
pivot_df = df.pivot(index="Model", columns="Platform", values="Accuracy")

pivot_df.plot(kind="bar", figsize=(8,5))
plt.title("Model Accuracy Comparison (Zomato vs Swiggy)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

In [ ]:
# -------------------------
# 0. Mount Google Drive & Import
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# -------------------------
# 1. Load Data (add platform here)
# -------------------------
zomato_df = pd.read_csv("/content/drive/MyDrive/Dissertation/New/zomato_reviews.csv")
zomato_df['platform'] = 'Zomato'

swiggy_df = pd.read_csv("/content/drive/MyDrive/Dissertation/New/swiggy_reviews.csv")
swiggy_df['platform'] = 'Swiggy'

# Combine
df = pd.concat([zomato_df, swiggy_df], ignore_index=True)

# -------------------------
# 2. Sentiment Labeling
# -------------------------
df['sentiment'] = df['score'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
sentiment_labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
df['sentiment_label'] = df['sentiment'].map(sentiment_labels)

# -------------------------
# 3. Sentiment Percentage Table (Fixed)
# -------------------------
sentiment_counts = df.groupby(['platform', 'sentiment_label']).size().reset_index(name='count')
total_counts = df.groupby('platform').size().reset_index(name='total')
sentiment_pct = pd.merge(sentiment_counts, total_counts, on='platform')
sentiment_pct['percentage'] = 100 * sentiment_pct['count'] / sentiment_pct['total']
sentiment_pct = sentiment_pct.drop(['count', 'total'], axis=1)

print("\nSentiment Percentage Table:")
print(tabulate(sentiment_pct, headers='keys', tablefmt='grid', showindex=False))

# -------------------------
# 4. Sentiment Percentage Graph
# -------------------------
plt.figure(figsize=(8,6))
sns.barplot(data=sentiment_pct, x='sentiment_label', y='percentage', hue='platform')
plt.title("Sentiment Percentage by Platform")
plt.ylabel("Percentage %")
plt.xlabel("Sentiment")
plt.ylim(0, 100)
plt.legend(title="Platform")
plt.tight_layout()
plt.show()

# -------------------------
# 5. Rating Percentage Table (Fixed)
# -------------------------
rating_counts = df.groupby(['platform', 'score']).size().reset_index(name='count')
total_counts = df.groupby('platform').size().reset_index(name='total')
rating_pct = pd.merge(rating_counts, total_counts, on='platform')
rating_pct['percentage'] = 100 * rating_pct['count'] / rating_pct['total']
rating_pct = rating_pct.drop(['count', 'total'], axis=1)

print("\nRating Percentage Table:")
print(tabulate(rating_pct, headers='keys', tablefmt='grid', showindex=False))

# -------------------------
# 6. Rating Percentage Graph
# -------------------------
plt.figure(figsize=(8,6))
sns.barplot(data=rating_pct, x='score', y='percentage', hue='platform')
plt.title("Rating Percentage by Platform")
plt.ylabel("Percentage %")
plt.xlabel("Rating (1-5)")
plt.ylim(0, 100)
plt.legend(title="Platform")
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import re
import emoji

# -------------------------
# 1. Load Data
# -------------------------
zomato_df = pd.read_csv("/content/drive/MyDrive/Dissertation/New/zomato_reviews.csv")
swiggy_df = pd.read_csv("/content/drive/MyDrive/Dissertation/New/swiggy_reviews.csv")

# -------------------------
# 2. Preprocessing Function
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
zomato_df['clean_content'] = zomato_df['content'].apply(clean_text)
swiggy_df['clean_content'] = swiggy_df['content'].apply(clean_text)

# -------------------------
# 3. Word Count
# -------------------------
zomato_df['word_count'] = zomato_df['clean_content'].apply(lambda x: len(x.split()))
swiggy_df['word_count'] = swiggy_df['clean_content'].apply(lambda x: len(x.split()))

# -------------------------
# 4. Filter exactly 10 words & take first 10 samples
# -------------------------
zomato_sample = zomato_df[zomato_df['word_count'] == 10].head(10)
swiggy_sample = swiggy_df[swiggy_df['word_count'] == 10].head(10)

# Add Sr No.
zomato_sample.insert(0, 'Sr No.', range(1, 1 + len(zomato_sample)))
swiggy_sample.insert(0, 'Sr No.', range(1, 1 + len(swiggy_sample)))

# -------------------------
# 5. Print Tables
# -------------------------
def print_table(df, title, columns):
    print(f"\n{'='*50}")
    print(f"{title.upper()}")
    print(f"{'='*50}")
    print(df[columns].to_string(index=False))
    print(f"{'='*50}")

# Zomato Raw Data
print_table(zomato_sample,
            "Zomato Raw Reviews",
            ['Sr No.', 'content', 'score'])

# Zomato Preprocessed Data
print_table(zomato_sample,
            "Zomato Preprocessed Reviews",
            ['Sr No.', 'clean_content', 'score'])

# Swiggy Raw Data
print_table(swiggy_sample,
            "Swiggy Raw Reviews",
            ['Sr No.', 'content', 'score'])

# Swiggy Preprocessed Data
print_table(swiggy_sample,
            "Swiggy Preprocessed Reviews",
            ['Sr No.', 'clean_content', 'score'])